In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 0.3 Numerical Integration and Differentiation

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume 0 — Mathematical & Computational Foundations",
    number="0.3",
    title="Numerical Integration and Differentiation",
    blurb="Computing integrals and derivatives that cannot be done by hand: the "
    "quadrature rules, their convergence orders, Richardson "
    "extrapolation, and why raising the order beats shrinking the step.",
    difficulty="intermediate",
    estimate="80–110 min",
)

## Notebook overview

The mathematical symbols $\int_a^b f\,dx$ and $f'(x)$ are familiar to us. Yet the
question remains regarding exactly how a computer produces them when there is no
antiderivative to write down and no symbol to differentiate, which is almost
always, in real physics. The answers
are **quadrature rules** (integrals as weighted sums of samples) and **finite-
difference stencils** (derivatives as weighted differences), and the interesting
part is not the formulas but their **error behaviour**: how fast each converges,
and where round-off ([§0.1](floating-point.ipynb)) sets a floor that no smaller
step can beat.

The throughline is **order**. A rule's convergence order $p$ (error $\sim
C\,h^p$) is the single number that decides whether we need ten samples or ten
thousand. We will measure $p$ for the trapezoid, midpoint, and Simpson rules
(2, 2, 4); double it for free with **Richardson extrapolation**; reach the
spectacular efficiency of **Gaussian quadrature** ($n$ points exact for degree
$2n-1$); measure the orders of three derivative stencils (1, 2, 4); and close the
loop with [§0.1](floating-point.ipynb): for a smooth function, **raising the order beats shrinking the
step**, because higher order reaches a lower error *before* round-off takes over.

This points forward: adaptive quadrature (Exercise 8) is the workhorse behind
the orbital and scattering integrals of
[§2.4](../02-classical-mechanics/central-force.ipynb)–[§2.5](../02-classical-mechanics/scattering.ipynb)
and the descent-time integral of
[§2.8](../02-classical-mechanics/brachistochrone.ipynb). There are no
animations here: nothing in this material *moves*; a
convergence plot is best read as a still figure, and that is the right choice.

> **How to read the checks.** Each exercise ends with a `validate` call against
> an independent fact: a known integral, a predicted order, an exactness result.
> A ✓ is strong evidence; a ✗ is a prompt to *locate the discrepancy* (an even-$n$
> requirement, an order fit on the round-off tail), not a verdict.

> **Scope.** A working review, not a numerical-analysis text. The standard
> reference is Press et al., *Numerical Recipes*, ch. 4 {cite}`numrecipes`;
> round-off limits trace back to [§0.1](floating-point.ipynb) and Higham
> {cite}`higham2002`.

## Theory in brief

### Quadrature: integrals as weighted samples

A **quadrature rule** approximates $\int_a^b f\,dx$ by a weighted sum of function
values, and the rules differ only in *which* values and *what* weights. The
simplest family, **Newton–Cotes**, fixes equally spaced samples $x_i=a+ih$,
$h=(b-a)/n$, and reads the weights off the polynomial that interpolates them, so
a better interpolant gives a better rule. A straight line through the panel
endpoints yields the **composite trapezoid** rule,

```{math}
:label: eq-trap
T(h) = h\left[\tfrac12 f_0 + f_1 + \cdots + f_{n-1} + \tfrac12 f_n\right],
\qquad \text{error } = O(h^2);
```

sampling the panel *centres* instead gives the **composite midpoint** rule, also
$O(h^2)$ but typically with about half the error, since its over- and undershoots
on a convex arc partly cancel:

```{math}
:label: eq-mid
M(h) = h\sum_{i=0}^{n-1} f\!\left(a + (i+\tfrac12)h\right), \qquad \text{error } = O(h^2).
```

Stepping up from a line to a *parabola* across each pair of panels gives
**Simpson's rule**, where one extra sample per pair of panels buys a remarkable **two
orders** (all three error terms follow from the Taylor remainder of the
interpolating polynomial; Press et al., *Numerical Recipes*, §4.1, tabulate
them in full):

```{math}
:label: eq-simpson
S(h) = \frac{h}{3}\left[f_0 + 4f_1 + 2f_2 + 4f_3 + \cdots + 4f_{n-1} + f_n\right],
\qquad \text{error } = O(h^4) \quad (n \text{ even}).
```

### Order, Richardson, and Gauss

The **order** $p$ of a rule is defined by error $\approx C\,h^p \approx C'\,n^{-p}$,

```{math}
:label: eq-quad-order
\varepsilon(n) \approx C'\,n^{-p} \quad\Longrightarrow\quad
\log\varepsilon \approx \text{const} - p\,\log n,
```

so $p$ is **minus the slope** of a log–log error-vs-$n$ plot. **Richardson
extrapolation** combines two estimates to *cancel* the leading error term: since
trapezoid error is $\propto h^2$, the combination

```{math}
:label: eq-richardson
\frac{4\,T(h/2) - T(h)}{3} = O(h^4)
```

is two orders better for free; iterating this is **Romberg** integration. (The
Euler–Maclaurin formula supplies the underlying $h^2$ series; Press et al.,
*Numerical Recipes*, §§4.2–4.3, develop it and build Romberg from it.)
**Gaussian quadrature** goes further still: by choosing *both* the nodes and the
weights optimally (the nodes are roots of Legendre polynomials), $n$ points
integrate every polynomial of degree $\le 2n-1$ **exactly**,

```{math}
:label: eq-gauss
\int_{-1}^{1} f(x)\,dx \approx \sum_{i=1}^{n} w_i\,f(x_i), \qquad
\text{exact for } \deg f \le 2n-1,
```

and converges *spectrally* (faster than any fixed power of $n$) for smooth $f$.
The exactness theorem rests on the orthogonality of the Legendre polynomials;
Press et al., *Numerical Recipes*, §4.6, give the proof.

### Differentiation stencils and the round-off floor

Derivatives combine samples with differencing weights. The **forward**,
**central**, and **fourth-order central** stencils are

```{math}
:label: eq-stencils
\begin{aligned}
f'(x) &\approx \frac{f(x+h)-f(x)}{h} && O(h), \\[2pt]
f'(x) &\approx \frac{f(x+h)-f(x-h)}{2h} && O(h^2), \\[2pt]
f'(x) &\approx \frac{-f(x+2h)+8f(x+h)-8f(x-h)+f(x-2h)}{12h} && O(h^4).
\end{aligned}
```

Each row follows from Taylor-expanding $f(x\pm h)$ about $x$ and cancelling
the unwanted terms; Press et al., *Numerical Recipes*, §5.7, tabulate the
standard stencils. As in [§0.1](floating-point.ipynb), shrinking $h$ helps only
until round-off in the differenced
numerator takes over; past that the error *grows*. The lever for a smooth
function is therefore **order, not step**: a higher-order stencil reaches a
lower error *before* hitting its round-off floor.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import erf
from scipy.integrate import quad
from numpy.polynomial.legendre import leggauss

from ecp import validate


def trapezoid(f, a, b, n):
    """Composite trapezoid rule (eq-trap of the theory section).

    Second-order quadrature: straight-line panels, the baseline Newton–Cotes rule.

    Parameters
    ----------
    f : callable
        Integrand.
    a, b : float
        Limits.
    n : int
        Number of panels.

    Returns
    -------
    float
        The approximate integral.
    """
    x = np.linspace(a, b, n + 1)
    y = f(x)
    h = (b - a) / n
    return h * (0.5 * y[0] + y[1:-1].sum() + 0.5 * y[-1])


def midpoint(f, a, b, n):
    """Composite midpoint rule (eq-mid of the theory section).

    Also second order, evaluating the integrand at panel centres.

    Parameters
    ----------
    f : callable
        Integrand.
    a, b : float
        Limits.
    n : int
        Number of panels.

    Returns
    -------
    float
        The approximate integral.
    """
    h = (b - a) / n
    xm = a + (np.arange(n) + 0.5) * h
    return h * f(xm).sum()


def simpson(f, a, b, n):
    """Composite Simpson rule (eq-simpson of the theory section).

    Fourth order from parabolic panels — the order jump over trapezoid/midpoint.

    Parameters
    ----------
    f : callable
        Integrand.
    a, b : float
        Limits.
    n : int
        Number of panels (even).

    Returns
    -------
    float
        The approximate integral.
    """
    if n % 2:
        n += 1  # Simpson needs an even panel count (parabolas span PAIRS of
        # panels); rounding up silently beats raising in a teaching context, but the
        # caller should know the effective n may differ by one
    x = np.linspace(a, b, n + 1)
    y = f(x)
    h = (b - a) / n
    return h / 3 * (y[0] + y[-1] + 4 * y[1:-1:2].sum() + 2 * y[2:-1:2].sum())


def fit_order(ns, errs):
    """Empirical convergence order p: minus the slope of log(error) vs log(n).

    Parameters
    ----------
    ns : array_like
        Panel counts.
    errs : array_like
        Corresponding errors.

    Returns
    -------
    float
        The estimated order $p$.
    """
    ns, errs = np.asarray(ns, float), np.asarray(errs, float)
    # Exclude points at the round-off floor (0.1): once the error saturates near
    # machine epsilon it no longer follows C·n^(−p), and including those points
    # would drag the fitted slope toward zero.
    m = errs > 1e-13
    return -np.polyfit(np.log(ns[m]), np.log(errs[m]), 1)[0]


# Smooth test integral used throughout: ∫₀¹ eˣ dx = e − 1.
f_exp = np.exp
I_exp = np.e - 1.0

## Exercise 1 — Trapezoid and midpoint from scratch

The two simplest quadrature rules come from
replacing $f$ on each panel by something integrable: a straight line through the
endpoints gives the **trapezoid** rule {eq}`eq-trap`, and a flat line through the
panel centre gives the **midpoint** rule {eq}`eq-mid`. Both are $O(h^2)$, but they
err in *opposite* directions (trapezoid over-counts a convex arc, midpoint under-
counts it), and the midpoint error is typically about half the trapezoid's.
{numref}`fig-quad-panels` shows the three constructions on our running
integrand $e^x$.

1. Implement both rules (the `trapezoid` and `midpoint` helpers above).
2. Test them on $\int_0^1 e^x\,dx=e-1$ at a coarse and a finer panel count,
   watching the error shrink.

In [ ]:
# (solution hidden on the public site)


### Solution — Exercise 1

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    trapezoid(f_exp, 0, 1, 8),
    I_exp,
    "trapezoid rule approximates ∫₀¹eˣ (coarse)",
    rtol=1e-2,
)
validate.close(
    trap_estimate, I_exp, "trapezoid rule converges as panels are added", rtol=1e-4
)

## Exercise 2 — Convergence order of Newton–Cotes (worked)

The order $p$ in error $\approx C'n^{-p}$,
{eq}`eq-quad-order`, is what makes a rule worth using: it is **minus the slope**
of a log–log error-vs-$n$ plot. Trapezoid and midpoint are $p=2$; Simpson is
$p=4$. This exercise *measures* those slopes directly.

1. For $\int_0^1 e^x\,dx$, compute the error of each rule over a geometric sweep
   of $n$.
2. Plot the errors log–log ({numref}`fig-quad-convergence`).
3. Fit the slopes with the `fit_order` helper (`numpy.polyfit` in log space) and
   compare to the theoretical orders 2, 2, 4.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(p_trap, 2.0, "trapezoid is second order", atol=0.15)
validate.close(p_mid, 2.0, "midpoint is second order", atol=0.15)
validate.close(p_simpson, 4.0, "Simpson is fourth order", atol=0.2)

## Exercise 3 — Simpson's rule and the order jump

The jump from $O(h^2)$ to $O(h^4)$ at Simpson's rule
{eq}`eq-simpson` is the best bargain in elementary quadrature: replacing each
straight-line panel by a *parabola* costs one extra sample per pair of panels but
squares the error's dependence on $h$. At a fixed, modest $n$ the difference is
already dramatic.

1. For the integral $\int_0^1 e^x\,dx$ (exact value $e-1$), compute the
   trapezoid and Simpson errors at the same small $n=8$ subintervals.
2. Confirm Simpson is orders of magnitude better at identical cost.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.check(
    e_simp8 < 1e-2 * e_trap8,
    "Simpson's parabolic panels gain orders over trapezoid at the same n",
    f"Simpson {e_simp8:.2e} vs trapezoid {e_trap8:.2e}",
)

## Exercise 4 — Richardson extrapolation → Romberg

Because the trapezoid error is a clean power series
in $h^2$, two estimates at $h$ and $h/2$ contain enough information to *cancel*
the leading $O(h^2)$ term. The combination $\big(4T(h/2)-T(h)\big)/3$,
{eq}`eq-richardson`, is $O(h^4)$ (Simpson's rule in disguise) and iterating the
cancellation up a triangular table is **Romberg** integration, which reaches very
high accuracy from a handful of trapezoid evaluations.

1. For the integral $\int_0^1 e^x\,dx$ (exact value $e-1$), form the Richardson
   combination $\big(4T(h/2)-T(h)\big)/3$ from the trapezoid estimates at $n=16$
   and $n=32$, and show it is far more accurate than either input.
2. Build a small Romberg table and watch the accuracy climb column by column.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    richardson_estimate,
    I_exp,
    "Richardson extrapolation of trapezoid is far more accurate",
    rtol=1e-6,
)

## Exercise 5 — Gaussian quadrature: the 2n−1 magic

Newton–Cotes fixes the nodes at equal spacing.
**Gaussian quadrature** frees them: with $n$ nodes *and* $n$ weights chosen
optimally (the nodes turn out to be the roots of the degree-$n$ Legendre
polynomial), the rule integrates every polynomial up to degree $2n-1$ **exactly**
({eq}`eq-gauss`). So $n=3$ points nail any quintic, and for smooth functions the
error falls *spectrally*, faster than any power of $n$. We map the standard
interval $[-1,1]$ to $[a,b]$ by $x\mapsto \tfrac{b-a}{2}x+\tfrac{a+b}{2}$.

1. Integrate the polynomial $p(x)=3x^5-2x^3+x-1$, whose exact integral over
   $[0,1]$ is $-1/2$, with 3-point Gauss (`numpy.polynomial.legendre.leggauss`
   for the nodes and weights) and confirm it is exact.
2. For the smooth integral $\int_0^1 e^x\,dx$ (exact $e-1$), compare Gauss and
   Simpson errors versus the number of points ({numref}`fig-quad-gauss`).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    gauss3_on_deg5, exact_poly, "3-point Gauss is exact for degree ≤ 5", atol=1e-12
)

## Exercise 6 — Numerical differentiation stencils

Derivatives are built the same way: combine nearby
samples with weights that cancel the unwanted Taylor terms. The **forward**
difference {eq}`eq-stencils` keeps only the first term and is $O(h)$; the
**central** difference cancels the second-order term by symmetry and is $O(h^2)$;
the **fourth-order central** stencil combines four points to cancel through
$O(h^4)$. More points, higher order.

1. Implement the three stencils of {eq}`eq-stencils`.
2. Measure their orders by differentiating $f=\sin$ at $x=1$ (so $f'=\cos 1$ is
   the exact target), fitting the error-vs-$h$ slopes (`numpy.polyfit` in log
   space) over the truncation-dominated regime ({numref}`fig-quad-stencils`).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(p_forward, 1.0, "forward difference is first order", atol=0.15)
validate.close(p_central, 2.0, "central difference is second order", atol=0.15)
validate.close(p_4th, 4.0, "the 4th-order stencil is fourth order", atol=0.2)

## Exercise 7 — Round-off revisited: order beats step (callback to 0.1)

[§0.1](floating-point.ipynb) showed a single derivative's error is
U-shaped in $h$: truncation $\sim h^p$ falling, round-off $\sim \varepsilon/h$
rising, with a minimum at an intermediate $h_\star$. The order $p$ sets *how low*
that minimum reaches. So for a smooth function the way to a more accurate
derivative is not a smaller step (round-off caps that) but a **higher-order
stencil**, whose U bottoms out **lower, and at a larger (safer) $h$**.

1. Sweep $h$ across many decades (`numpy.logspace`) for the forward and central
   stencils, and plot both U-curves on one axis ({numref}`fig-quad-ucurve`).
2. Confirm the central difference reaches a lower minimum error than the forward
   difference — and reaches it at a larger, safer step.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.check(
    min_error_central < min_error_forward,
    "the higher-order stencil reaches a lower error floor",
    f"central {min_error_central:.2e} < forward {min_error_forward:.2e}",
)

## Exercise 8 — Adaptive quadrature in practice (synthesis)

Production integrators do not use a fixed grid: they
**place samples adaptively**, refining only where the integrand varies, and
**self-estimate** their error. `scipy.integrate.quad` (adaptive Gauss–Kronrod) is
the workhorse, and it is exactly what the course leans on for the orbital and
scattering integrals of
[§2.4](../02-classical-mechanics/central-force.ipynb)–[§2.5](../02-classical-mechanics/scattering.ipynb)
and the descent-time integral of
[§2.8](../02-classical-mechanics/brachistochrone.ipynb). On a
sharply peaked integrand, a fixed grid of the same size misses the spike entirely
while the adaptive rule clusters its points there and nails it.

1. Integrate the narrow Gaussian spike
   $f(x)=\exp\!\big[-(x-0.5)^2/(2\cdot 0.03^2)\big]$ on $[0,1]$ with
   `scipy.integrate.quad`, against the closed form
   $0.03\sqrt{2\pi}\,\operatorname{erf}\!\big(0.5/(0.03\sqrt2)\big)\approx0.0752$
   (`math.erf`).
2. Contrast with a fixed trapezoid grid spending a comparable number of
   evaluations, which misses the spike.

In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.close(
    quad_value,
    peak_exact,
    "adaptive quadrature matches the known integral of a sharp peak",
    rtol=1e-8,
)

## Notebook summary

- Newton–Cotes quadrature with measured orders: trapezoid and midpoint second order, Simpson
  fourth; Richardson extrapolation building Romberg; and Gaussian quadrature exact to degree
  $2n-1$.
- Finite-difference stencils (forward first order, central second, a fourth-order stencil) and
  the round-off-versus-truncation tradeoff (callback to
  [§0.1](floating-point.ipynb)); adaptive quadrature in practice.

## Outlook

- **Singular and improper integrals.** Endpoint singularities and infinite ranges
  are tamed by variable transformations: the tanh–sinh ("double exponential")
  rule is a remarkably robust default.
- **Many dimensions.** Tensor-product quadrature suffers the *curse of
  dimensionality* ($n^d$ points); **Monte Carlo** integration trades order for
  dimension-independent $O(N^{-1/2})$ error: a forward link to the statistical
  mechanics of Volume V.
- **Spectral methods.** Clenshaw–Curtis quadrature and **FFT-based
  differentiation** achieve exponential accuracy for periodic/smooth functions
  (a forward link to [§0.6](fft.ipynb)).
- **Automatic differentiation.** The exact alternative to finite differences:
  derivatives to machine precision with no step to tune, and the engine behind
  modern machine learning and gradient-based physics.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()